# Phase V2: vec2vec seed and data sweep (two runs at once, one per T4)

Neural Nexus, vector-DB compression project (team leader: Aman).

| | |
|---|---|
| **Time** | about 11 hours (10.5 h of training + setup). Kaggle's limit is 12 h |
| **Settings** | Accelerator = **GPU T4 x2**, Internet = **On**, Persistence = off |
| **Inputs** | FRIEND 1-4: Add Input -> Notebooks -> `phaseA_fp32_embeddings` and `phaseA2b_verify_attacker_extra` (Aman shares both with you). FRIEND 5: no inputs. FRIEND 6-7: only `phaseA_fp32_embeddings` |
| **The only line you change** | `FRIEND = ...` in cell 2 (Aman tells you your number) |
| **How to run** | first `QUICK_TEST = True` + Run All (about 10 min, checks everything works); then `QUICK_TEST = False` and **Save Version -> Save & Run All (Commit)**; close the tab |
| **Output** | `runs/<name>/`: log, curve.png, summary.json, resume.pt (to continue the run on the workstation), translators |

What it does: trains the authors' official vec2vec (commit 5efb3c0, their settings) twice at the same time,
one run per GPU. Each run differs only in its seed (the GAN's random starting point) or in its data.
The authors warn that their GAN often fails to start depending on the seed; this sweep looks for a seed and
data setting where it does start. Early stopping is OFF (in our first run the authors' rule stopped training
after about 6% of the training the paper uses). Every 30 minutes a full checkpoint is saved.

When it is finished: Share -> add `amanpriyansh` as **Can view**, and tell Aman.

In [ ]:
# Cell 1 - install + official vec2vec code at a fixed commit (2-4 minutes)
!mkdir -p /kaggle/working/code /kaggle/working/runs
!pip install -q "transformers==4.46.3" "vec2text==0.0.13" "sentence-transformers==3.4.1" accelerate toml wandb evaluate diffusers nltk seaborn
!rm -rf /kaggle/working/vec2vec && git clone -q https://github.com/rjha18/vec2vec.git /kaggle/working/vec2vec && cd /kaggle/working/vec2vec && git checkout -q 5efb3c02ee908847581c6488cf9c8ed6dace3c29 && git log -1 --format='vec2vec commit %H %cd'

In [ ]:
# Cell 2 - settings. CHANGE ONLY THESE TWO LINES.
FRIEND = 6            # 1-7 (Aman tells you which). 6 and 7 = same-source attacker runs (8 Nov)
QUICK_TEST = False    # True = 10-minute check on a tiny subset; False = the real 10.5-hour run

# The sweep. Each friend runs two lines, one per GPU. (The workstation runs ours fp32 seeds 7 and 8.)
SWEEP = {
    1: [dict(kind="ours", seed=5), dict(kind="ours", seed=1)],
    2: [dict(kind="ours", seed=2), dict(kind="ours", seed=3)],
    3: [dict(kind="ours", seed=4), dict(kind="ours", seed=6)],
    4: [dict(kind="dpr_matched", seed=5), dict(kind="dpr_matched", seed=1)],
    5: [dict(kind="native", seed=5, num_points=1_000_000), dict(kind="native", seed=5, num_points=50_000)],
    6: [dict(kind="ours_ss", seed=3), dict(kind="ours_ss", seed=4)],   # added 8 Nov: same-source attacker set,
    7: [dict(kind="ours_ss", seed=6), dict(kind="ours_ss", seed=7)],   # near-copies removed (workstation: seeds 1, 2, 5)
}
BUDGET_HOURS = {"ours": 10.5, "ours_ss": 10.5, "dpr_matched": 10.5, "native": 10.0}   # native epochs are long; keep room under 12 h

import os, sys, json, time, glob, subprocess
import numpy as np, torch
assert torch.cuda.device_count() == 2, "Set Accelerator = GPU T4 x2 (two GPUs), then restart"
RUNS = []
for spec in SWEEP[FRIEND]:
    name = spec["kind"] + (f"{spec['num_points'] // 1000}k" if spec["kind"] == "native" else "")
    name += f"_fp32_s{spec['seed']}" if spec["kind"] in ("ours", "ours_ss") else f"_s{spec['seed']}"
    RUNS.append({**spec, "name": name + ("_quicktest" if QUICK_TEST else "")})
for gpu, r in enumerate(RUNS):
    print(f"GPU {gpu}: {r['name']}   budget {0.05 if QUICK_TEST else BUDGET_HOURS[r['kind']]} h")

In [ ]:
# Cell 3 - what we expect BEFORE the run (written down so the result cannot move the goalposts)
EXPECT = {
    "random guess": "top-1 = 1/8,192 = 0.00012, average rank = 4,096 (ours / dpr_matched); "
                    "authors' top-1 within 1,024 = 0.001 (native)",
    "steps per epoch": "ours 195, dpr_matched 390, native-1M 3,906, native-50k 195 (batch 256)",
    "minutes per epoch on a T4": "ours about 0.7-2 (unknown before this run; the workstation needed 0.66 with two "
                                 "runs sharing a faster GPU); native much longer because it also runs two text encoders",
    "training steps in 10.5 h": "ours roughly 60,000-180,000. Our first run had only 19,000. The paper reports seeds "
                                "that converged within 10 epochs of 1M points, about 39,000 steps",
    "LEARNED (ours, dpr_matched)": "top-1 >= 0.01 (80x random) OR average rank <= a quarter of random (1,024 of 8,192) at any epoch",
    "LEARNED (native)": "authors' validation top-1 (within 1,024) >= 0.1 at any epoch",
    "chance that at least one of our 8 seeds learns (6 here + 2 on the workstation)": "unknown. If each seed had the paper's cross-backbone rate "
        "(e5->gtr: 3 of 15 seeds), about 1 - 0.8^8 = 0.83. The paper also reports gte->gtr failing without "
        "per-pair tuning, so zero is a real possibility and would itself be a result",
    "native-1M (positive control)": "should learn: it is the authors' own pair and pipeline. If it does not, the budget "
                                    "per run is too short and a failure of our pair is inconclusive",
    "native-50k vs native-1M": "if 1M learns and 50k does not, too little data is the problem (Aman's hypothesis)",
    "dpr_matched vs ours": "if dpr_matched learns and ours does not, more / matched victim-side data helps",
}
for k, v in EXPECT.items(): print(f"- {k}: {v}")

In [ ]:
%%writefile /kaggle/working/code/v2v_runner.py
"""Run the OFFICIAL vec2vec trainer (github.com/rjha18/vec2vec, commit 5efb3c0).

Two modes:
  * precomputed (default): the trainer gets OUR saved vectors through its own "precomputed embeddings"
    path (the one the authors use for fMRI data). Victim side = BGE vectors (fp32 or decoded compressed),
    attacker side = GTR vectors. No pairs are given to training.
  * native=True: the authors' own text pipeline and default model pair from configs/unsupervised.toml
    (positive control: does vec2vec learn at all on this machine in this time?).

What is changed in the authors' code, and nothing else:
  1. Data input (precomputed mode only) and per-space encoder sizes (ours are 1024 and 768).
  2. A wall-clock budget (training time, summed over resumed sessions) instead of the epoch limit.
  3. Optional: early stopping switched off. Their rule (min 80 epochs, patience 20) counts epochs and was
     calibrated for 1M-vector epochs (~3,900 steps); our epochs are ~195 steps, so it stopped run 1 after
     ~6% of the paper's minimum training.
  4. After every epoch (precomputed mode) we score with OUR metric, same as mini-vec2vec: top-1, average
     rank and mean cosine of translated victim vectors vs their true GTR vectors (true vectors used only here).
  5. Full checkpoints (translator, all discriminators, all optimisers and schedulers, random states) every
     30 minutes and at the end, so a run can be resumed on another machine or session.
  6. wandb logging replaced by a local JSON-lines log.
All losses, architecture, optimisers and hyper-parameters come from the authors' configs/unsupervised.toml
unless the caller overrides them explicitly through extra_args (and then they are written in the log).
"""
import os, sys, json, time, types, inspect

def run(repo_dir, X_victim, Y_attacker, X_eval, Y_eval, out_dir, budget_hours,
        seed=5, val_size=4096, extra_args=None, log=print,
        native=False, early_stopping=True, resume_from=None, ckpt_every_minutes=30):
    import numpy as np, torch
    t_start = time.time()
    out_dir = os.path.abspath(out_dir)
    resume_from = os.path.abspath(resume_from) if resume_from else None
    repo_dir = os.path.abspath(repo_dir)
    os.makedirs(out_dir, exist_ok=True)
    sys.path.insert(0, repo_dir)
    os.chdir(repo_dir)

    # beir is only used by dataset paths we do not use; stub it so it need not be installed
    if "beir" not in sys.modules:
        for name in ["beir", "beir.datasets", "beir.datasets.data_loader_hf"]:
            sys.modules[name] = types.ModuleType(name)
        sys.modules["beir.datasets.data_loader_hf"].HFDataLoader = object

    # Windows: vec2text imports the Unix-only "resource" module; give it a harmless stand-in
    if os.name == "nt" and "resource" not in sys.modules:
        try:
            import resource  # noqa: F401  (already provided, e.g. by a shim)
        except ImportError:
            stub = types.ModuleType("resource")
            stub.RLIMIT_NOFILE = 7
            stub.getrlimit = lambda *_: (8192, 8192)
            stub.setrlimit = lambda *_: None
            sys.modules["resource"] = stub

    import wandb
    import train as T
    from fmri.data import FmriSingleViewDataset

    # Windows: data-loader worker processes use "spawn" and would have to copy the whole dataset;
    # their helper also uses a Linux-only call. Load data in the main process instead.
    if os.name == "nt":
        T.get_num_proc = lambda: 0
    if not torch.cuda.is_available():          # their helper divides by the GPU count (CPU-only test runs)
        T.get_num_proc = lambda: 2

    # -- resume metadata (states are loaded into the live objects at the first epoch)
    ckpt = None
    if resume_from:
        ckpt = torch.load(resume_from, map_location="cpu", weights_only=False)
        log(f"resuming from {resume_from}: epoch {ckpt['epoch']}, {ckpt['train_hours']:.2f} training hours so far")

    # -- 6. local logging instead of wandb
    log_path = os.path.join(out_dir, "v2v_log.jsonl")
    def _append(path, rec):
        with open(path, "a") as f:
            f.write(json.dumps(rec) + "\n")
    def _wandb_log(d, *a, **k):
        rec = {"t_hours": (time.time() - t_start) / 3600, "train_hours": _train_hours()}
        for key, v in d.items():
            if isinstance(v, (int, float)):
                rec[key] = float(v)
        _safe(lambda: _append(log_path, {"kind": "authors_val", **rec}), "v2v_log.jsonl")
    wandb.log = _wandb_log
    wandb.Image = lambda *a, **k: None

    # -- 2. training-time budget, summed over resumed sessions; at least one epoch always runs
    clock = {"t0": None, "offset": ckpt["train_hours"] if ckpt else 0.0}
    def _train_hours():
        return clock["offset"] + (0.0 if clock["t0"] is None else (time.time() - clock["t0"]) / 3600)
    def _time_up():
        if clock["t0"] is None:
            clock["t0"] = time.time()
            return False
        return _train_hours() > budget_hours

    # -- 1. our data through the precomputed path
    if not native:
        Xv = torch.tensor(np.asarray(X_victim, dtype=np.float32))
        Ya = torch.tensor(np.asarray(Y_attacker, dtype=np.float32))
        Xe = torch.tensor(np.asarray(X_eval, dtype=np.float32))
        Ye = torch.tensor(np.asarray(Y_eval, dtype=np.float32))

        class PairedDifferentSizes(torch.utils.data.Dataset):
            """Same as the authors' FmriPairedDataset, but the two spaces may have different sizes."""
            def __init__(self, name_a, Za, name_b, Zb):
                assert len(Za) == len(Zb)
                self.a, self.Za, self.b, self.Zb = name_a, Za, name_b, Zb
            def __len__(self):
                return len(self.Za)
            def __getitem__(self, i):
                return {self.a: self.Za[i], self.b: self.Zb[i]}

        def _loader(*a, **k):
            supset = FmriSingleViewDataset("gtr", Ya)          # attacker's own space ("supervised" = known model)
            unsupset = FmriSingleViewDataset("bge", Xv)        # stolen victim vectors ("unsupervised" = unknown)
            n = min(val_size, len(Xe))
            val = PairedDifferentSizes("gtr", Ye[:n], "bge", Xe[:n])
            return supset, unsupset, val, {"gtr": Ya.shape[1], "bge": Xv.shape[1]}
        T.load_fmri_train_and_eval_datasets = _loader
        Xe_n = torch.nn.functional.normalize(Xe, dim=1)
        Ye_n = torch.nn.functional.normalize(Ye, dim=1)

    # -- 4. our metric after every epoch
    def _score(translator):
        dev = next(translator.parameters()).device
        was_training = translator.training
        translator.eval()
        outs = []
        with torch.no_grad():
            for i in range(0, len(Xe_n), 1024):
                ins = {"bge": Xe_n[i:i+1024].to(dev), "gtr": Ye_n[i:i+1024].to(dev)}
                _, tr = translator(ins, include_reps=False)
                outs.append(tr["gtr"]["bge"].float().cpu())
        if was_training:
            translator.train()
        Z = torch.nn.functional.normalize(torch.cat(outs), dim=1)
        S = Z @ Ye_n.T
        dg = S.diag()
        ranks = (S > dg[:, None]).sum(1) + 1
        return {"top1": float((ranks == 1).float().mean()), "avg_rank": float(ranks.float().mean()),
                "mean_cos": float(dg.mean())}

    # -- 5. full checkpoints
    def _safe(fn, what, tries=6):
        """Saving must never kill a multi-day run: on Windows a backup program reading the file can block
        overwriting it for a moment. Retry, then log and carry on (the next save will try again)."""
        for i in range(tries):
            try:
                fn(); return True
            except OSError as err:
                if i == tries - 1:
                    log(f"WARNING: could not write {what} ({err}); training continues, will retry at the next save")
                    return False
                time.sleep(5)
    GAN_NAMES = ["gan", "sup_gan", "latent_gan", "similarity_gan"]
    def _sd(x):
        return x.state_dict() if x is not None and hasattr(x, "state_dict") else None
    def _save_ckpt(k):
        acc = k["accelerator"]
        blob = {"epoch": state["epoch"], "train_hours": _train_hours(), "best": dict(best),
                "translator": acc.unwrap_model(k["translator"]).state_dict(),
                "gen_optimizers": [o.state_dict() for o in k["gen_optimizers"]],
                "gen_schedulers": [s.state_dict() for s in k["gen_schedulers"]],
                "gans": {n: {"disc": _sd(acc.unwrap_model(getattr(k[n], "discriminator", None))
                                         if getattr(k[n], "discriminator", None) is not None else None),
                             "opt": _sd(getattr(k[n], "discriminator_opt", None)),
                             "sched": _sd(getattr(k[n], "discriminator_scheduler", None))}
                         for n in GAN_NAMES if k.get(n) is not None},
                "scaler": _sd(getattr(acc, "scaler", None)),
                "rng": {"torch": torch.get_rng_state(),
                        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
                        "numpy": np.random.get_state()},
                "args": args}
        tmp = os.path.join(out_dir, "resume.pt.tmp")
        ok = _safe(lambda: torch.save(blob, tmp), "resume.pt.tmp") and \
             _safe(lambda: os.replace(tmp, os.path.join(out_dir, "resume.pt")), "resume.pt")  # never half-written
        state["last_ckpt"] = time.time()
        if ok:
            log(f"checkpoint saved at epoch {state['epoch']} ({_train_hours():.2f} training hours)")
    def _load_ckpt(k):
        acc = k["accelerator"]
        acc.unwrap_model(k["translator"]).load_state_dict(ckpt["translator"])
        for o, sd in zip(k["gen_optimizers"], ckpt["gen_optimizers"]): o.load_state_dict(sd)
        for s, sd in zip(k["gen_schedulers"], ckpt["gen_schedulers"]): s.load_state_dict(sd)
        for n, g in ckpt["gans"].items():
            obj = k.get(n)
            if obj is None: continue
            if g["disc"] is not None: acc.unwrap_model(obj.discriminator).load_state_dict(g["disc"])
            if g["opt"] is not None: obj.discriminator_opt.load_state_dict(g["opt"])
            if g["sched"] is not None: obj.discriminator_scheduler.load_state_dict(g["sched"])
        if ckpt.get("scaler") is not None and getattr(acc, "scaler", None) is not None:
            acc.scaler.load_state_dict(ckpt["scaler"])
        torch.set_rng_state(ckpt["rng"]["torch"])
        if ckpt["rng"]["cuda"] is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(ckpt["rng"]["cuda"])
        np.random.set_state(ckpt["rng"]["numpy"])
        log("resume: all model, optimiser and scheduler states loaded")

    best = dict(ckpt["best"]) if ckpt else {"top1": -1.0}
    state = {"epoch": ckpt["epoch"] if ckpt else 0, "loaded": ckpt is None, "last_ckpt": time.time()}
    orig_loop = T.training_loop_
    def _loop(*a, **k):
        if not state["loaded"]:
            _load_ckpt(k); state["loaded"] = True
        t0 = time.time()
        r = orig_loop(*a, **k)
        state["epoch"] += 1
        rec = {"kind": "ours", "epoch": state["epoch"], "t_hours": (time.time() - t_start) / 3600,
               "train_hours": _train_hours(), "epoch_minutes": (time.time() - t0) / 60}
        if not native:
            sc = _score(k["translator"])
            rec.update(sc)
        _safe(lambda: _append(log_path, rec), "v2v_log.jsonl")
        log(json.dumps(rec))
        model = k["accelerator"].unwrap_model(k["translator"])
        _safe(lambda: torch.save(model.state_dict(), os.path.join(out_dir, "translator_latest.pt")), "translator_latest.pt")
        if not native and sc["top1"] > best["top1"]:
            best.update(sc, epoch=state["epoch"])
            _safe(lambda: torch.save(model.state_dict(), os.path.join(out_dir, "translator_best.pt")), "translator_best.pt")
            _safe(lambda: json.dump(best, open(os.path.join(out_dir, "best.json"), "w")), "best.json")
        if (time.time() - state["last_ckpt"]) / 60 >= ckpt_every_minutes or _train_hours() > budget_hours:
            _save_ckpt(k)
        return r
    T.training_loop_ = _loop

    # -- patch main(): per-space encoder sizes, time budget, drop_last flag, logging call
    src = inspect.getsource(T.main)
    a = "encoder_dims = {cfg.sup_emb: int(embed_dim), cfg.unsup_emb: int(embed_dim)}"
    b = "encoder_dims = {cfg.sup_emb: int(embed_dim[cfg.sup_emb]), cfg.unsup_emb: int(embed_dim[cfg.unsup_emb])}"
    c = "    for epoch in range(max_num_epochs):\n"
    d = c + "        if _TIME_UP():\n            print('Time budget reached, stopping.')\n            break\n"
    e = "drop_last=cfg.train_drop_last if hasattr(cfg, 'train_drop_last') else not fmri_mode"
    f_ = "drop_last=bool(cfg.train_drop_last) if hasattr(cfg, 'train_drop_last') else not fmri_mode"
    g = "wandb.log(val_res)"   # wandb.init() re-binds wandb.log, so route the call explicitly
    assert src.count(a) == 1 and src.count(c) == 1 and src.count(e) == 2 and src.count(g) == 1, \
        "vec2vec code changed: patch points not found"
    src = src.replace(a, b).replace(c, d).replace(e, f_).replace(g, "_WANDB_LOG(val_res)")
    T.__dict__["_WANDB_LOG"] = _wandb_log
    T.__dict__["_TIME_UP"] = _time_up
    exec(compile(src, T.__file__, "exec"), T.__dict__)

    if native:
        args = {"epochs": 100000, "use_wandb": 0, "seed": seed,
                "save_dir": out_dir.rstrip("/\\") + "/authors_{}/", "force_wandb_name": 1, "wandb_name": "run"}
    else:
        args = {"unsup_emb": "bge", "sup_emb": "gtr", "dataset": "fmri_precomputed", "epochs": 100000,
                "use_wandb": 0, "seed": seed, "val_size": val_size,
                "save_dir": out_dir.rstrip("/\\") + "/authors_{}/", "force_wandb_name": 1, "wandb_name": "run",
                # our two sides have different sizes: drop the last partial batch so every step pairs two
                # full batches, as the authors' text path does by default
                "train_drop_last": 1}
    if not early_stopping:
        args["min_epochs"] = 10 ** 9      # their early-stopping check only starts after min_epochs
    args.update(extra_args or {})
    if ckpt is not None:   # a resumed run must be the same experiment
        old = {k: v for k, v in ckpt["args"].items() if k != "save_dir"}
        new = {k: v for k, v in args.items() if k != "save_dir"}
        assert old == new, f"resume: settings differ from the checkpoint\n old {old}\n new {new}"
    with open(log_path, "a") as f:
        f.write(json.dumps({"kind": "run_args", "native": native, "early_stopping": early_stopping,
                            "resume_from": resume_from, "budget_hours": budget_hours, **args}) + "\n")
    argv = ["train.py", "unsupervised"]
    for k_, v_ in args.items():
        argv += [f"--{k_}", str(v_)]
    sys.argv = argv
    T.argv = argv
    T.main()
    return best, log_path

In [ ]:
%%writefile /kaggle/working/code/v2v_job.py
"""One vec2vec run (one variant, one seed). Used by the Kaggle sweep notebook (two of these at once, one per T4)
and by the workstation (to continue the winning run from its checkpoint). Neural Nexus vector-DB project.

Kinds:
  ours         victim = 50k BGE-large vectors (fp32 or decoded compressed), attacker = 200k A2 GTR vectors.
               This is the experiment of the paper.
  dpr_matched  data diagnostic: victim side = BGE-large of A2 texts 0-99,999 (embedded here), attacker side =
               GTR of A2 texts 100,000-199,999. Twice the victim-side data, both sides from the same text pool.
               Scored on the same 8,192 victim passages as "ours".
  ours_ss      same as ours, but the attacker side is the 19,569 SAME-SOURCE GTR vectors from Phase A
               (gtr_att_L32.npy) instead of the 200k DPR set. Added 8 Nov: mini-vec2vec only worked with this set.
  native       positive control: the authors' own pair (stella -> gte) and own NQ text pipeline, num_points per side.
               Scored only with the authors' validation metrics (their 4,096 held-out NQ passages).

Usage:
  python v2v_job.py --kind ours --scheme fp32 --seed 5 --hours 10.5 --data /kaggle/input --out runs/ours_s5 --repo vec2vec
  python v2v_job.py ... --resume runs/ours_s5/resume.pt --hours 60      (continue: --hours is the new TOTAL budget)
"""
import os, sys, json, time, glob, argparse, platform

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
# Windows: matplotlib defaults to the Tk backend; the authors' validation heatmaps (only for wandb, unused here)
# leak Tk bitmaps and after ~800 epochs Tk aborts the process ("Fail to allocate bitmap"). Draw off-screen instead.
os.environ.setdefault("MPLBACKEND", "Agg")
p = argparse.ArgumentParser()
p.add_argument("--kind", required=True, choices=["ours", "ours_ss", "dpr_matched", "native"])
p.add_argument("--scheme", default="fp32", help="ours only: fp32 or a Phase B scheme such as pq128x8")
p.add_argument("--seed", type=int, required=True)
p.add_argument("--hours", type=float, required=True, help="total training-time budget (summed over resumed sessions)")
p.add_argument("--data", required=True, help="folder searched (recursively) for the input files")
p.add_argument("--out", required=True)
p.add_argument("--repo", required=True, help="clone of github.com/rjha18/vec2vec at commit 5efb3c0")
p.add_argument("--resume", default=None, help="resume.pt from an earlier session of THE SAME run")
p.add_argument("--num_points", type=int, default=1_000_000, help="native only: texts per side")
p.add_argument("--early_stopping", type=int, default=0, help="1 = keep the authors' rule (min 80 epochs, patience 20)")
p.add_argument("--smoke", action="store_true", help="tiny subset, to test the setup")
a = p.parse_args()

import numpy as np, torch
from v2v_runner import run

OUT = os.path.abspath(a.out)
os.makedirs(OUT, exist_ok=True)
REPO = os.path.abspath(a.repo)
if os.name == "nt":   # keep Windows awake while this process runs (no admin rights needed)
    import ctypes
    ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | 0x00000001)

L, MODEL, EVAL_N = 32, "bge-large", 8192
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"kind={a.kind} scheme={a.scheme} seed={a.seed} gpu={gpu} budget={a.hours}h out={OUT}", flush=True)
assert torch.cuda.is_available() or a.smoke, "No CUDA GPU visible to PyTorch"

def find_one(name):
    hits = sorted(set(glob.glob(os.path.join(a.data, "**", name), recursive=True)))
    assert len(hits) >= 1, f"input file {name} not found under {a.data}"
    if len(hits) > 1:   # the same file can be attached through two notebooks; it must be identical
        sizes = {os.path.getsize(h) for h in hits}
        assert len(sizes) == 1, f"{name}: several different copies found: {hits}"
    return hits[0]

@torch.no_grad()
def embed_bge(texts, bs=128):
    """Identical to Phase A (build_A.py cell 4): fp16 BGE-large, CLS token, L2-normalised, no prefix for passages."""
    from transformers import AutoModel, AutoTokenizer
    tok = AutoTokenizer.from_pretrained("BAAI/bge-large-en-v1.5")
    m = AutoModel.from_pretrained("BAAI/bge-large-en-v1.5", torch_dtype=torch.float16).to("cuda").eval()
    out = []
    for i in range(0, len(texts), bs):
        b = tok(texts[i:i + bs], padding=True, truncation=True, max_length=512, return_tensors="pt").to("cuda")
        h = m(**b).last_hidden_state[:, 0]
        out.append(torch.nn.functional.normalize(h.float(), dim=-1).cpu())
    del m; torch.cuda.empty_cache()
    return torch.cat(out).numpy().astype(np.float32)

extra = {} if torch.cuda.is_available() else {"mixed_precision": "no"}
info = {}
if a.kind in ("ours", "ours_ss", "dpr_matched"):
    Y_true = np.load(find_one(f"gtr_victim_L{L}.npy")).astype(np.float32)          # scoring only
    X_vic_fp32 = np.load(find_one(f"{MODEL}_victim_L{L}.npy")).astype(np.float32)
    assert X_vic_fp32.shape == (50000, 1024) and Y_true.shape == (50000, 768)
    eval_idx = np.random.default_rng(0).choice(50000, EVAL_N, replace=False)       # same 8,192 as every phase
    if a.kind == "ours_ss":
        Y_att_all = np.load(find_one(f"gtr_att_L{L}.npy")).astype(np.float32)
        assert Y_att_all.shape[1] == 768 and len(Y_att_all) > 15000 and np.isfinite(Y_att_all).all()
        # same filter as Phase C2 v3: drop attacker passages that are near-copies (centred GTR cosine > 0.95)
        # of any victim passage (NQ repeats paragraphs; 1.56% of victim passages had one)
        _m = Y_att_all.mean(0)
        _c = lambda Z: torch.nn.functional.normalize(torch.from_numpy(Z - _m), dim=1)
        _a, _v = _c(Y_att_all), _c(Y_true)
        _amx = torch.cat([(_a[i:i + 4096] @ _v.T).max(1).values for i in range(0, len(_a), 4096)]).numpy()
        Y_att_all = Y_att_all[_amx <= 0.95]
        print(f"near-copy filter: kept {len(Y_att_all)} of {len(_amx)} attacker passages", flush=True)
        del _a, _v
    else:
        Y_att_all = np.load(find_one(f"gtr_attextra_L{L}.npy")).astype(np.float32)
        assert Y_att_all.shape == (200000, 768) and np.isfinite(Y_att_all).all()
    if a.kind in ("ours", "ours_ss"):
        X_train = X_vic_fp32 if a.scheme == "fp32" else \
            np.load(find_one(f"decoded_victim_{a.scheme}_{MODEL}_L{L}.npy")).astype(np.float32)
        assert X_train.shape == (50000, 1024) and np.isfinite(X_train).all()
        X_eval = X_train[eval_idx]
        Y_train = Y_att_all
    else:
        cache = os.path.join(OUT, f"bge-large_attextra0-100k_L{L}.npy")
        if os.path.exists(cache):
            X_train = np.load(cache)
        else:
            texts = json.load(open(find_one("texts_extra.json")))[f"extra_p_L{L}"]
            assert len(texts) == 200000
            t = time.time(); X_train = embed_bge(texts[:100000]); np.save(cache, X_train)
            print(f"embedded 100,000 A2 texts with BGE-large in {time.time() - t:.0f}s", flush=True)
        assert X_train.shape == (100000, 1024) and abs(np.linalg.norm(X_train, axis=1).mean() - 1) < 1e-3
        Y_train = Y_att_all[100000:]                  # other texts: the two sides share no passage
        X_eval = X_vic_fp32[eval_idx]
    Y_eval = Y_true[eval_idx]
    del Y_true, X_vic_fp32, Y_att_all            # keep only what training needs (two runs share the machine's RAM)
    X_train, Y_train = [z.copy() if z.base is not None else z for z in (X_train, Y_train)]  # drop views of big arrays
    if a.smoke:
        X_train, Y_train, X_eval, Y_eval = X_train[:5000], Y_train[:20000], X_eval[:1024], Y_eval[:1024]
        extra.update({"val_bs": 512, "top_k_size": 512, "top_k_batches": 2})
    info = {"n_victim_side": len(X_train), "n_attacker_side": len(Y_train), "n_eval": len(X_eval),
            "steps_per_epoch": min(len(X_train), len(Y_train)) // 256}
    print(json.dumps(info), flush=True)
    native = False
else:
    X_train = Y_train = X_eval = Y_eval = None
    extra.update({"num_points": 2000 if a.smoke else a.num_points})
    if a.smoke:
        extra.update({"val_size": 512, "val_bs": 256, "top_k_size": 256, "top_k_batches": 2})
    info = {"num_points_per_side": extra["num_points"], "steps_per_epoch": extra["num_points"] // 256}
    native = True

t0 = time.time()
best, log_path = run(REPO, X_train, Y_train, X_eval, Y_eval, OUT, budget_hours=a.hours, seed=a.seed,
                     val_size=min(4096, len(X_eval)) if X_eval is not None else 4096,
                     extra_args=extra, log=lambda s: print(s, flush=True), native=native,
                     early_stopping=bool(a.early_stopping), resume_from=a.resume)
os.chdir(HERE)

# ---- summary + plot
rows = [json.loads(l) for l in open(log_path)]
ours = [r for r in rows if r["kind"] == "ours"]
val = [r for r in rows if r["kind"] == "authors_val"]
def authors_top1(r):
    """Authors' top-1 (within batches of top_k_size = 1,024 candidates, averaged over top_k_batches) for the
    unknown -> known direction: the victim ("unsupervised") space translated into the attacker's space."""
    unsup, sup = ("stella", "gte") if native else ("bge", "gtr")
    keys = [k for k in r if k.startswith(f"val/{unsup}_{sup}_top_1_acc")]
    return max((r[k] for k in keys), default=float("nan"))
summary = {"kind": a.kind, "scheme": a.scheme, "seed": a.seed, "budget_hours": a.hours, "smoke_test": a.smoke,
           "resumed_from": a.resume, "machine": platform.node(), "gpu": gpu, "torch": torch.__version__, **info,
           "train_hours": ours[-1]["train_hours"], "epochs": ours[-1]["epoch"],
           "steps": ours[-1]["epoch"] * info["steps_per_epoch"],
           "minutes_per_epoch_median": float(np.median([r["epoch_minutes"] for r in ours])),
           "authors_val_top1_last": authors_top1(val[-1]) if val else None,
           "authors_val_top1_max": max(authors_top1(r) for r in val) if val else None}
if not native:
    summary.update({"final": {k: ours[-1][k] for k in ["top1", "avg_rank", "mean_cos"]},
                    "best_by_true_pairs": best,
                    "min_avg_rank_any_epoch": min(r["avg_rank"] for r in ours),
                    "max_top1_any_epoch": max(r["top1"] for r in ours),
                    "note": "final = honest unsupervised result; best = picked with the true pairs (upper bound)"})
json.dump(summary, open(os.path.join(OUT, "summary.json"), "w"), indent=1)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
name = f"{a.kind} {a.scheme if a.kind in ('ours', 'ours_ss') else ''} seed {a.seed}".replace("  ", " ")
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
if not native:
    t = [r["train_hours"] for r in ours]
    ax[0].plot(t, [r["top1"] for r in ours], color="tab:red", label="top-1 (our metric)")
    ax[0].axhline(1 / len(X_eval), color="gray", ls="--", lw=1, label="random guess")
    ax[0].set_ylabel(f"Top-1 accuracy ({len(X_eval):,} candidates)")
    ax[1].plot(t, [r["avg_rank"] for r in ours], color="tab:blue", label="average rank (our metric)")
    ax[1].axhline((len(X_eval) + 1) / 2, color="gray", ls="--", lw=1, label="random guess")
    ax[1].set_ylabel(f"Average rank of the true vector (1 = perfect)")
    ax[1].set_title(f"vec2vec {name}: average rank vs time")
else:
    t = [r["train_hours"] for r in val]
    ax[0].plot(t, [authors_top1(r) for r in val], color="tab:red", label="top-1 (authors' validation)")
    ax[0].set_ylabel("Top-1 accuracy (authors' validation set)")
    ax[1].plot([r["train_hours"] for r in ours], [r["epoch_minutes"] for r in ours], color="tab:green",
               label="minutes per epoch")
    ax[1].set_ylabel("Minutes per epoch")
    ax[1].set_title(f"vec2vec {name}: speed")
ax[0].set_title(f"vec2vec {name}: top-1 vs time")
for x in ax:
    x.set_xlabel(f"Training time (hours, {gpu})"); x.legend(); x.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "curve.png"), dpi=130)
print("DONE", json.dumps(summary), flush=True)

In [ ]:
# Cell 6 - check inputs and download the models / texts once (so the two runs do not download at the same time)
os.makedirs("/kaggle/working/runs", exist_ok=True)
kinds = {r["kind"] for r in RUNS}
if kinds & {"ours", "ours_ss", "dpr_matched"}:
    need = ["bge-large_victim_L32.npy", "gtr_victim_L32.npy"] + \
           (["gtr_attextra_L32.npy"] if kinds & {"ours", "dpr_matched"} else ["gtr_att_L32.npy"]) + \
           (["texts_extra.json"] if "dpr_matched" in kinds else [])
    for f in need:
        hits = glob.glob(f"/kaggle/input/**/{f}", recursive=True)
        assert hits, f"missing input {f}: Add Input -> Notebooks -> phaseA_fp32_embeddings and phaseA2b_verify_attacker_extra"
        print("found", hits[0])
if "dpr_matched" in kinds:
    from huggingface_hub import snapshot_download
    snapshot_download("BAAI/bge-large-en-v1.5")
if "native" in kinds:
    from datasets import load_dataset
    from sentence_transformers import SentenceTransformer
    t = time.time()
    ds = load_dataset("jxm/nq_corpus_dpr", split="train"); print(f"NQ DPR corpus: {len(ds):,} passages ({time.time()-t:.0f}s)")
    assert len(ds) >= 2 * 1_000_000 + 4096
    for m in ["infgrad/stella-base-en-v2", "thenlper/gte-base"]:
        SentenceTransformer(m, trust_remote_code=True); print("downloaded", m)
    del ds

In [ ]:
# Cell 7 - start both runs (one per GPU) and print progress every 10 minutes. This is the long cell.
procs = []
for gpu, r in enumerate(RUNS):
    out = f"/kaggle/working/runs/{r['name']}"
    os.makedirs(out, exist_ok=True)
    cmd = [sys.executable, "-u", "/kaggle/working/code/v2v_job.py", "--kind", r["kind"], "--seed", str(r["seed"]),
           "--hours", str(0.05 if QUICK_TEST else BUDGET_HOURS[r["kind"]]), "--data", "/kaggle/input",
           "--out", out, "--repo", "/kaggle/working/vec2vec"]
    if r["kind"] == "native": cmd += ["--num_points", str(r["num_points"])]
    if QUICK_TEST: cmd += ["--smoke"]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    procs.append((r, out, subprocess.Popen(cmd, env=env, stdout=open(f"{out}/stdout.txt", "w"), stderr=subprocess.STDOUT)))
    time.sleep(60)   # stagger the starts

def last_line(out):
    try:
        rows = [json.loads(l) for l in open(f"{out}/v2v_log.jsonl")]
    except FileNotFoundError:
        return "starting"
    ep = [x for x in rows if x["kind"] == "ours"]
    if not ep: return "setting up / first epoch"
    e = ep[-1]
    if "top1" in e:
        return (f"epoch {e['epoch']}  {e['train_hours']:.2f} h  top1 {e['top1']:.4f}  avg_rank {e['avg_rank']:.0f}"
                f"  (best rank so far {min(x['avg_rank'] for x in ep):.0f})")
    val = [x for x in rows if x["kind"] == "authors_val"]
    t1 = [v for k, v in val[-1].items() if k.startswith("val/stella_gte_top_1_acc")] if val else []
    return f"epoch {e['epoch']}  {e['train_hours']:.2f} h  authors' top-1 {t1[0] if t1 else float('nan'):.4f}"

t0 = time.time()
while any(p.poll() is None for _, _, p in procs):
    time.sleep(30 if QUICK_TEST else 600)
    print(f"--- {(time.time()-t0)/3600:.2f} h since start")
    for r, out, p in procs:
        state = "running" if p.poll() is None else f"ended (code {p.returncode})"
        print(f"  {r['name']:24s} {state:16s} {last_line(out)}")
for r, out, p in procs:
    if p.returncode != 0:
        why = " (killed from outside: usually out of memory)" if p.returncode in (-9, 137) else ""
        print(f"\n!!! {r['name']} FAILED with exit code {p.returncode}{why}. Last lines of its stdout.txt:")
        print("".join(open(f"{out}/stdout.txt").readlines()[-30:]))

In [ ]:
# Cell 8 - results: verdict per run (rule fixed in cell 3) and the curves
from IPython.display import Image, display
rows = []
for r, out, p in procs:
    if not os.path.exists(f"{out}/summary.json"):
        rows.append({"run": r["name"], "status": "FAILED, see stdout.txt"}); continue
    s = json.load(open(f"{out}/summary.json"))
    if r["kind"] == "native":
        learned = (s["authors_val_top1_max"] or 0) >= 0.1
        rows.append({"run": r["name"], "hours": round(s["train_hours"], 2), "epochs": s["epochs"], "steps": s["steps"],
                     "authors_top1_last": s["authors_val_top1_last"], "authors_top1_max": s["authors_val_top1_max"],
                     "LEARNED": learned})
    else:
        learned = s["max_top1_any_epoch"] >= 0.01 or s["min_avg_rank_any_epoch"] <= (s["n_eval"] + 1) / 2 / 4
        rows.append({"run": r["name"], "hours": round(s["train_hours"], 2), "epochs": s["epochs"], "steps": s["steps"],
                     "min_per_epoch": round(s["minutes_per_epoch_median"], 2),
                     "top1_final": s["final"]["top1"], "avg_rank_final": round(s["final"]["avg_rank"]),
                     "top1_max": s["max_top1_any_epoch"], "avg_rank_min": round(s["min_avg_rank_any_epoch"]),
                     "LEARNED": learned})
import pandas as pd
table = pd.DataFrame(rows); print(table.to_string(index=False))
json.dump({"friend": FRIEND, "quick_test": QUICK_TEST, "results": rows, "expected": EXPECT},
          open("/kaggle/working/runs/sweep_result.json", "w"), indent=1, default=str)
for r, out, p in procs:
    if os.path.exists(f"{out}/curve.png"): display(Image(f"{out}/curve.png"))
import shutil
shutil.rmtree("/kaggle/working/vec2vec", ignore_errors=True)   # do not keep the code copy in the output
for r, out, p in procs:
    print(r["name"], sorted(os.listdir(out)))
if QUICK_TEST:
    print("\nQUICK TEST PASSED" if all(p.returncode == 0 for _, _, p in procs) else "\nQUICK TEST FAILED - send Aman the error above")
    print("Now set QUICK_TEST = False and do Save Version -> Save & Run All (Commit).")

## Reading the result (for Aman)

- `LEARNED` uses the rule written in cell 3 before the run.
- `top1_final` is the honest number (what an attacker gets without the answers). `top1_max` / `avg_rank_min`
  look at every epoch with the true pairs, so they are only used to decide whether the GAN started learning.
- `resume.pt` lets the workstation continue exactly this run (same seed, same optimiser state) for days.